In [1]:
import pandas as pd
import sys, os
from pathlib import Path

import os
import math
sys.path.append(str(Path(os.getcwd()).parent))
from utils.wrappers import measure_time_and_space, measure_time
from typing import Iterable, List
from utils.data_structures import RollingMeanArray, UpwardsDownwardsArray

### Getting list of filepaths

In [2]:
@measure_time_and_space
def select_target_csvs(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = {
        "MSFT.csv",
        "NVDA.csv",
        "AAPL.csv",
        "GOOGL.csv",
        "AMZN.csv",
        "META.csv",
        "TSLA.csv",
    }

    selected = []
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file() and entry.name in target_files:
                selected.append(entry.path)
    return selected

@measure_time_and_space
def select_target_csvs_nonrecursive(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = [
        "MSFT.csv",
        "NVDA.csv",
        "AAPL.csv",
        "GOOGL.csv",
        "AMZN.csv",
        "META.csv",
        "TSLA.csv",
        ]
    

    selected_paths = []

    # Efficiently iterate through directory (not recursive)
    for filename in os.listdir(directory):
        if filename in target_files:
            selected_paths.append(os.path.join(directory, filename))

    return selected_paths

In [3]:
select_target_csvs(Path.cwd() / "csv")
select_target_csvs_nonrecursive(Path.cwd() / "csv")



[select_target_csvs] Time elapsed: 0.001040 seconds
[select_target_csvs] Peak memory: 3.24 KB
[select_target_csvs_nonrecursive] Time elapsed: 0.000547 seconds
[select_target_csvs_nonrecursive] Peak memory: 38.16 KB


['c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\AAPL.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\AMZN.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\GOOGL.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\META.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\MSFT.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\NVDA.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\TSLA.csv']

In [4]:
file_paths = select_target_csvs(Path.cwd() / "csv")

[select_target_csvs] Time elapsed: 0.001042 seconds
[select_target_csvs] Peak memory: 3.24 KB


### Reading the selected csv files in a efficient manner

In [5]:
@measure_time_and_space

def read_filtered_csvs(file_paths, chunksize=100000, min_year = 2025, max_year = 2025):
    """
    Reads multiple CSV files in chunks, filters rows with Date.year > 2022,
    and returns a single combined DataFrame.
    """
    combined_chunks = []  # list to store filtered chunks

    for path in file_paths:
        for chunk in pd.read_csv(path, parse_dates=['Date'], chunksize=chunksize, usecols= lambda x: x not in ["Adj Close"]):
            filtered_chunk = chunk[(chunk['Date'].dt.year >= min_year) & (chunk['Date'].dt.year <= max_year)] # Filter rows where year > 2024
            if not filtered_chunk.empty:
                combined_chunks.append(filtered_chunk)

    # Concatenate all filtered chunks into a single DataFrame
    combined_df = pd.concat(combined_chunks, ignore_index=True) 
    return combined_df

combined_df = read_filtered_csvs(file_paths)

[read_filtered_csvs] Time elapsed: 0.141142 seconds
[read_filtered_csvs] Peak memory: 4509.68 KB


In [6]:
combined_df.tail()

,Date,Ticker,Open,High,Low,Close,Volume
2765,2025-10-02,TSLA,470.540009,470.750000,435.570007,436.000000,137009000
2766,2025-10-03,TSLA,443.290009,446.769989,416.579987,429.829987,133188200
2767,2025-10-06,TSLA,440.750000,453.549988,436.690002,453.250000,85324900
2768,2025-10-07,TSLA,447.820007,452.679993,432.450012,433.089996,101798300
2769,2025-10-08,TSLA,437.757996,437.769989,425.230011,433.149994,21848536


In [7]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2770 entries, 0 to 2769
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    2770 non-null   datetime64[ns]
 1   Ticker  2770 non-null   object        
 2   Open    2770 non-null   float64       
 3   High    2770 non-null   float64       
 4   Low     2770 non-null   float64       
 5   Close   2770 non-null   float64       
 6   Volume  2770 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 151.6+ KB


## This section is experimental and is only intended to compare the space time complexity of various algorithms
### Actual feature engineering will be done below

In [8]:
#Test Df so we can compare space time complexity between different implementations
test_df = combined_df[combined_df['Ticker'] == 'AAPL'].copy()

In [ ]:
test_df.head()

,Date,Ticker,Open,High,Low,Close,Volume
0,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700
1,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700
2,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100
3,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100
4,2025-01-06,AAPL,244.309998,247.330002,243.199997,245.000000,45045600


In [10]:
@measure_time_and_space
def sma_pandas(df, window=20):
    """
    Calculate Simple Moving Average (SMA) for the 'close' column.
    """
    return df['Close'].rolling(window=window).mean()

sma_pandas(test_df, window=30)

[sma_pandas] Time elapsed: 0.000499 seconds
[sma_pandas] Peak memory: 14.17 KB


0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
          ...    
391    256.160633
392    256.350966
393    256.496967
394    256.483967
395    256.523268
Name: Close, Length: 396, dtype: float64

In [11]:
rma = RollingMeanArray(test_df['Close'],30)

In [12]:
@measure_time_and_space
def sma_rma():
    """
    Calculate Simple Moving Average (SMA) for the 'close' column using RollingMeanArray class.
    """
    return pd.Series(rma.rolling_mean())
sma_rma()

[sma_rma] Time elapsed: 0.000509 seconds
[sma_rma] Peak memory: 35.86 KB


0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
          ...    
391    256.160633
392    256.350966
393    256.496967
394    256.483967
395    256.523268
Length: 396, dtype: float64

In [ ]:
@measure_time_and_space
def sma_naive():
    """
    Calculate Simple Moving Average (SMA) for the 'close' column using RollingMeanArray.naive_rolling_mean method, as a benchmark against the sma_rma method.
    """
    return pd.Series(rma.naive_rolling_mean())
test_df["SMA_30"] = sma_naive()

[sma_naive] Time elapsed: 0.000806 seconds
[sma_naive] Peak memory: 33.67 KB


In [ ]:
rma.window = 90
test_df["SMA_90"] = sma_rma()
rma.window = 180
test_df["SMA_180"] = sma_rma()

[sma_rma] Time elapsed: 0.000580 seconds
[sma_rma] Peak memory: 32.27 KB
[sma_rma] Time elapsed: 0.000282 seconds
[sma_rma] Peak memory: 30.11 KB


In [13]:
sma_rma()

IndexError: list index out of range

In [90]:
test_df

,Date,Ticker,Open,High,Low,Close,Volume,SMA_30,SMA_90,SMA_180
0,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700,NaN,NaN,NaN
1,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700,NaN,NaN,NaN
2,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100,NaN,NaN,NaN
3,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100,NaN,NaN,NaN
4,2025-01-06,AAPL,244.309998,247.330002,243.199997,245.000000,45045600,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
391,2025-10-02,AAPL,256.579987,258.179993,254.149994,257.130005,42630200,256.160633,241.778877,224.793439
392,2025-10-03,AAPL,254.669998,259.239990,253.949997,258.019989,49155600,256.350966,242.094099,225.093994
393,2025-10-06,AAPL,257.989990,259.070007,255.050003,256.690002,44664100,256.496967,242.394544,225.387161
394,2025-10-07,AAPL,256.809998,257.399994,255.429993,256.480011,31923700,256.483967,242.651766,225.692883


In [63]:
uda = UpwardsDownwardsArray(test_df['Close'])

In [64]:
@measure_time_and_space
def create_run_group():
    """
    Calculate the difference between consecutive 'close' prices in-place.
    """
    return uda.create_run_group()

@measure_time_and_space
def create_run_group_naive():
    """
    Calculate the difference between consecutive 'close' prices in-place.
    """
    return uda.create_run_group_naive()

test_df["Run_Group"] = create_run_group()
test_df["Run_Group"] = create_run_group_naive()



[create_run_group] Time elapsed: 0.000105 seconds
[create_run_group] Peak memory: 0.15 KB
[create_run_group_naive] Time elapsed: 0.000152 seconds
[create_run_group_naive] Peak memory: 3.25 KB


In [65]:
test_df

,Date,Ticker,Open,High,Low,Close,Volume,SMA_30,SMA_90,SMA_180,Run_Group
0,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700,NaN,NaN,NaN,NaN
1,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700,NaN,NaN,NaN,0.0
2,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100,NaN,NaN,NaN,-1.0
3,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100,NaN,NaN,NaN,1.0
4,2025-01-06,AAPL,244.309998,247.330002,243.199997,245.000000,45045600,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...
391,2025-10-02,AAPL,256.579987,258.179993,254.149994,257.130005,42630200,256.160633,241.778877,224.793439,1.0
392,2025-10-03,AAPL,254.669998,259.239990,253.949997,258.019989,49155600,256.350966,242.094099,225.093994,0.0
393,2025-10-06,AAPL,257.989990,259.070007,255.050003,256.690002,44664100,256.496967,242.394544,225.387161,-1.0
394,2025-10-07,AAPL,256.809998,257.399994,255.429993,256.480011,31923700,256.483967,242.651766,225.692883,0.0


In [ ]:
# from itertools import groupby
# f = []
# for key, group in groupby(run_group):
#     f.append((key, len(list(group))))
# f


[(nan, 1),
 (0, 1),
 (-1, 1),
 (1, 2),
 (-1, 2),
 (1, 2),
 (-1, 2),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 2),
 (-1, 2),
 (1, 2),
 (-1, 2),
 (1, 2),
 (-1, 2),
 (1, 1),
 (-1, 1),
 (1, 2),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 2),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 2),
 (-1, 2),
 (1, 2),
 (-1, 2),
 (1, 2),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 2),
 (1, 2),
 (-1, 1),
 (1, 1),
 (-1, 2),
 (1, 2),
 (-1, 2),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 2),
 (-1, 2),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 2),
 (-1, 2),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 2),
 (-1, 1),
 (1, 1),
 (-1, 2),
 (1, 2),
 (-1, 2),
 (1, 2),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 2),
 (1, 2),
 (-1, 2),
 (1, 2),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 2),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 1),
 (-1, 1),
 (1, 2),
 (-1, 2),
 (1, 2),
 (-1, 1),
 (1, 1),
 (-1, 2),
 (1, 1),
 (-1, 1),
 (1, 2),
 (-1, 2),
 

## Start of actual Feature Engineering


In [71]:
rolling_windows = [30, 90, 180]  # 1, 3, 6 months roughly

@measure_time_and_space
def process_group(group: pd.DataFrame) -> pd.DataFrame:
    close_vals = group["Close"].tolist()
    

    # 1️⃣ Compute multiple rolling means efficiently
    for w in rolling_windows:
        if len(close_vals) >= w:
            ma_vals = RollingMeanArray(close_vals, w).rolling_mean()
        else:
            ma_vals = [math.nan] * len(close_vals)
        group[f"MA_{w}"] = ma_vals

    # 2️⃣ Compute price differences -> Run Group
    group["Run_Group"] = UpwardsDownwardsArray(close_vals).create_run_group()

    return group


# Apply per company
final_df = combined_df.groupby("Ticker", group_keys=False, sort=False).apply(process_group)


[process_group] Time elapsed: 0.002656 seconds
[process_group] Peak memory: 64.87 KB
[process_group] Time elapsed: 0.002250 seconds
[process_group] Peak memory: 62.52 KB
[process_group] Time elapsed: 0.002357 seconds
[process_group] Peak memory: 62.58 KB
[process_group] Time elapsed: 0.002207 seconds
[process_group] Peak memory: 62.70 KB
[process_group] Time elapsed: 0.002226 seconds
[process_group] Peak memory: 62.86 KB
[process_group] Time elapsed: 0.002172 seconds
[process_group] Peak memory: 62.56 KB
[process_group] Time elapsed: 0.002180 seconds
[process_group] Peak memory: 62.86 KB


In [72]:
final_df

,Date,Ticker,Open,High,Low,Close,Volume,MA_30,MA_90,MA_180,Run_Group
0,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700,NaN,NaN,NaN,NaN
1,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700,NaN,NaN,NaN,0.0
2,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100,NaN,NaN,NaN,-1.0
3,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100,NaN,NaN,NaN,0.0
4,2025-01-06,AAPL,244.309998,247.330002,243.199997,245.000000,45045600,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2765,2025-10-02,TSLA,470.540009,470.750000,435.570007,436.000000,137009000,441.398664,388.061777,353.312443,-1.0
2766,2025-10-03,TSLA,443.290009,446.769989,416.579987,429.829987,133188200,440.966664,389.050554,354.060721,-1.0
2767,2025-10-06,TSLA,440.750000,453.549988,436.690002,453.250000,85324900,441.315330,390.299554,354.939110,1.0
2768,2025-10-07,TSLA,447.820007,452.679993,432.450012,433.089996,101798300,441.638663,391.340777,355.630832,-1.0


In [73]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2770 entries, 0 to 2769
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       2770 non-null   datetime64[ns]
 1   Ticker     2770 non-null   object        
 2   Open       2770 non-null   float64       
 3   High       2770 non-null   float64       
 4   Low        2770 non-null   float64       
 5   Close      2770 non-null   float64       
 6   Volume     2770 non-null   int64         
 7   MA_30      2567 non-null   float64       
 8   MA_90      2147 non-null   float64       
 9   MA_180     1517 non-null   float64       
 10  Run_Group  2763 non-null   float64       
dtypes: datetime64[ns](1), float64(8), int64(1), object(1)
memory usage: 259.7+ KB


In [75]:
final_df.to_csv("mag7_stocks.csv", index=False)